# Лабораторная работа №3  
# RAG: Retrieval-Augmented Generation

---

**Выполнил(а):** студент(ка) группы ______  
**ФИО:** _______________________________  
**Дата:** _______________________________

---

## 1. Цель работы

После выполнения этой лабораторной работы вы:
- Понять проблему «галлюцинаций» у больших языковых моделей
- Изучить архитектуру RAG (Retrieval-Augmented Generation)
- Создать эмбеддинги для текстовых документов
- Освоить векторный поиск с помощью FAISS
- Построить систему ответов на вопросы по документам

---

## 2. Теоретические сведения

### 2.1. Проблема «галлюцинаций» LLM

**Галлюцинации** — когда модель генерирует правдоподобную, но фактически неверную информацию.

**Причины:**
- Модель не имеет доступа к актуальным данным
- Отсутствие проверки фактов в процессе генерации
- Стремление дать «красивый» ответ вместо точного

**Пример галлюцинации:**
```
Вопрос: «Кто был президентом России в 2025 году?»
Ответ модели: «Дмитрий Медведев»  ← Выдумка!
```

### 2.2. Архитектура RAG

**RAG (Retrieval-Augmented Generation)** — подход, сочетающий поиск информации с генерацией ответа.

```
┌─────────────────┐       ┌────────────────┐       ┌─────────────┐
│   Вопрос        │ ──→   │   Поиск в      │ ──→   │  Генерация  │
│   пользователя  │       │    базе знаний │       │  ответа с   │
│                 │       │   (Retrieval)  │       │  контекстом │
└─────────────────┘       └────────────────┘       └─────────────┘
```

**Этапы:**
1. **Индексирование** — документы разбиваются на чанки, создаются эмбеддинги
2. **Поиск** — для вопроса находятся релевантные чанки
3. **Генерация** — LLM формирует ответ на основе найденного контекста

### 2.3. Embeddings (Векторные представления)

**Эмбеддинг** — плотный вектор, представляющий смысл текста в многомерном пространстве.

| Текст | Эмбеддинг (упрощённо) |
|-------|----------------------|
| «Кошка сидит на ковре» | [0.12, -0.45, 0.78, ...] |
| «Кот лежит на диване» | [0.15, -0.42, 0.75, ...] |

**Свойства:**
- Семантически близкие тексты имеют близкие векторы
- Косинусное сходство измеряет близость векторов
- Размерность: 384, 768, 1024 и более

### 2.4. Векторный поиск

| Метод | Описание | Скорость | Точность |
|-------|----------|----------|----------|
| **Exact Search** | Полный перебор | Медленно | 100% |
| **ANN (Approximate)** | Приближённый поиск | Быстро | 95-99% |
| **FAISS** | Библиотека от Meta | Очень быстро | 97-99% |
| **HNSW** | Графовый индекс | Быстро | 98% |

### 2.5. LangChain

**LangChain** — фреймворк для разработки приложений на основе LLM.

| Компонент | Назначение |
|-----------|------------|
| `Document Loaders` | Загрузка документов из файлов, URL |
| `Text Splitters` | Разбиение текста на чанки |
| `Embeddings` | Создание векторных представлений |
| `Vector Stores` | Хранение и поиск эмбеддингов |
| `Retrievers` | Извлечение релевантных документов |
| `Chains` | Композиция компонентов в пайплайны |

---

## 3. Задание

### 3.1. Базовый уровень (обязательно)

1. Загрузите текстовые документы (статьи, документацию)
2. Разбейте документы на чанки (текстовые фрагменты)
3. Создайте эмбеддинги с помощью предобученной модели
4. Постройте векторный индекс в FAISS
5. Реализуйте поиск релевантных чанков по запросу
6. Настройте генерацию ответа с использованием контекста

### 3.2. Продвинутый уровень (дополнительно)

- Реализуйте оценку качества поиска (Precision@K, Recall@K)
- Сравните разные embedding-модели по качеству и скорости
- Добавьте метаданные к чанкам (источник, дата, категория)

### 3.3. Индивидуальное задание

Создайте RAG-систему для своей предметной области:
- **Вариант A:** Поиск по технической документации
- **Вариант B:** Ответы на вопросы по учебным материалам
- **Вариант C:** Поиск информации в нормативных документах

Подготовьте мини-датасет (5-10 документов) и протестируйте систему.

---

## 4. Ход работы

### 4.1. Подготовка окружения

**Важно:** Для этой работы требуется GPU для ускорения работы с эмбеддингами.

In [1]:
# Проверка доступности GPU
import torch

print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU устройство: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU не доступен. Работа возможна на CPU, но медленнее.")

GPU доступен: True
GPU устройство: Tesla T4


In [2]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers -q

# Проверка версий
import langchain
import faiss

print(f"LangChain version: {langchain.__version__}")
print(f"FAISS version: {faiss.__version__}")

LangChain version: 1.2.12
FAISS version: 1.13.2


In [3]:
import torch

# Импорты для LangChain (обновлённые)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Настройка устройства
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


### 4.2. Создание тестовых документов

Для демонстрации создадим набор документов на русском языке.

In [4]:
# Создаём коллекцию документов
documents = [
    Document(
        page_content="""
        Машинное обучение — это подраздел искусственного интеллекта, 
        который изучает методы построения моделей, способных обучаться 
        на данных. Основные типы машинного обучения: обучение с учителем, 
        обучение без учителя и обучение с подкреплением. Обучение с учителем 
        включает классификацию и регрессию, где модель обучается на размеченных данных.
        """,
        metadata={"source": "ml_intro.txt", "topic": "машинное обучение"}
    ),
    Document(
        page_content="""
        Нейронные сети — это вычислительные системы, вдохновлённые биологическими 
        нейронными сетями мозга. Они состоят из слоёв нейронов: входной слой, 
        скрытые слои и выходной слой. Глубокое обучение использует многоуровневые 
        нейронные сети для решения сложных задач, таких как распознавание изображений 
        и обработка естественного языка.
        """,
        metadata={"source": "neural_networks.txt", "topic": "нейронные сети"}
    ),
    Document(
        page_content="""
        Обработка естественного языка (NLP) — это область искусственного интеллекта, 
        которая занимается взаимодействием компьютеров и человеческого языка. 
        Основные задачи NLP включают токенизацию, лемматизацию, распознавание 
        именованных сущностей, машинный перевод и анализ тональности текста.
        """,
        metadata={"source": "nlp_intro.txt", "topic": "NLP"}
    ),
    Document(
        page_content="""
        Компьютерное зрение — это область искусственного интеллекта, которая 
        обучает компьютеры интерпретировать и понимать визуальную информацию. 
        Применения включают распознавание лиц, автономные транспортные средства, 
        медицинскую диагностику и контроль качества на производстве.
        """,
        metadata={"source": "computer_vision.txt", "topic": "компьютерное зрение"}
    ),
    Document(
        page_content="""
        Большие языковые модели (LLM) — это нейронные сети, обученные на огромных 
        объёмах текстовых данных. Они способны генерировать связный текст, отвечать 
        на вопросы, переводить между языками и выполнять другие задачи обработки 
        языка. Примеры включают GPT, BERT, LLaMA и другие архитектуры.
        """,
        metadata={"source": "llm_overview.txt", "topic": "LLM"}
    ),
    Document(
        page_content="""
        Обучение с подкреплением — это тип машинного обучения, в котором агент 
        учится принимать решения, взаимодействуя со средой. Агент получает награды 
        или штрафы за свои действия и стремится максимизировать совокупную награду. 
        Применения включают игры, робототехнику и управление ресурсами.
        """,
        metadata={"source": "reinforcement_learning.txt", "topic": "обучение с подкреплением"}
    )
]

print(f"Создано документов: {len(documents)}")
print("\nТемы документов:")
for doc in documents:
    print(f"  - {doc.metadata['topic']} ({doc.metadata['source']})")

Создано документов: 6

Темы документов:
  - машинное обучение (ml_intro.txt)
  - нейронные сети (neural_networks.txt)
  - NLP (nlp_intro.txt)
  - компьютерное зрение (computer_vision.txt)
  - LLM (llm_overview.txt)
  - обучение с подкреплением (reinforcement_learning.txt)


### 4.3. Разбиение текста на чанки

Разбиваем документы на меньшие фрагменты для лучшего поиска.

In [5]:
# Настройка разбиения текста
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,           # Размер чанка в символах
    chunk_overlap=50,         # Перекрытие между чанками
    length_function=len,      # Функция подсчёта длины
    separators=["\n\n", "\n", ". ", " ", ""]  # Приоритет разбиения
)

# Разбиение документов
chunks = text_splitter.split_documents(documents)

print(f"Количество чанков после разбиения: {len(chunks)}")
print(f"\nПример чанка:")
print(f"Текст: {chunks[0].page_content[:150]}...")
print(f"Метаданные: {chunks[0].metadata}")

Количество чанков после разбиения: 14

Пример чанка:
Текст: Машинное обучение — это подраздел искусственного интеллекта, 
        который изучает методы построения моделей, способных обучаться...
Метаданные: {'source': 'ml_intro.txt', 'topic': 'машинное обучение'}


### 4.4. Создание эмбеддингов

Используем предобученную модель для создания векторных представлений.

In [6]:
# Загрузка модели эмбеддингов
# Используем мультиязычную модель для поддержки русского языка
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"Загрузка модели эмбеддингов: {embedding_model_name}")

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

print("Модель эмбеддингов загружена")

Загрузка модели эмбеддингов: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель эмбеддингов загружена


In [7]:
# Тестирование эмбеддингов
test_text = "Что такое машинное обучение?"
test_embedding = embeddings.embed_query(test_text)

print(f"\nТестовый текст: '{test_text}'")
print(f"Размерность эмбеддинга: {len(test_embedding)}")
print(f"Первые 10 значений: {[round(x, 4) for x in test_embedding[:10]]}")


Тестовый текст: 'Что такое машинное обучение?'
Размерность эмбеддинга: 384
Первые 10 значений: [-0.0321, -0.0272, -0.058, -0.0756, -0.0006, -0.0223, 0.0222, -0.0248, 0.0053, 0.006]


### 4.5. Построение векторного индекса FAISS

Создаём векторное хранилище для быстрого поиска.

In [8]:
# Создание векторного хранилища
print("Создание векторного индекса FAISS...")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Индекс создан! Количество векторов: {vectorstore.index.ntotal}")

Создание векторного индекса FAISS...
Индекс создан! Количество векторов: 14


In [9]:
# Сохранение индекса на диск
vectorstore.save_local("./faiss_index")
print("Индекс сохранён в ./faiss_index")

Индекс сохранён в ./faiss_index


### 4.6. Поиск релевантных документов

Реализуем поиск по векторному индексу.

In [10]:
# Функция поиска
def search_documents(query, top_k=3):
    """
    Поиск наиболее релевантных документов по запросу.
    
    Args:
        query: Текстовый запрос
        top_k: Количество возвращаемых результатов
    
    Returns:
        Список найденных документов с оценками релевантности
    """
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return results

# Тестирование поиска
test_query = "Как работают нейронные сети?"

print(f"Поисковый запрос: '{test_query}'")
print("="*60)

results = search_documents(test_query, top_k=3)

for i, (doc, score) in enumerate(results, 1):
    print(f"\n📌 Результат #{i} (релевантность: {1 - score:.4f})")
    print(f"Источник: {doc.metadata['source']}")
    print(f"Текст: {doc.page_content.strip()}")

Поисковый запрос: 'Как работают нейронные сети?'

📌 Результат #1 (релевантность: 0.7411)
Источник: neural_networks.txt
Текст: Нейронные сети — это вычислительные системы, вдохновлённые биологическими 
        нейронными сетями мозга. Они состоят из слоёв нейронов: входной слой,

📌 Результат #2 (релевантность: 0.2607)
Источник: neural_networks.txt
Текст: скрытые слои и выходной слой. Глубокое обучение использует многоуровневые 
        нейронные сети для решения сложных задач, таких как распознавание изображений

📌 Результат #3 (релевантность: -0.1085)
Источник: llm_overview.txt
Текст: Большие языковые модели (LLM) — это нейронные сети, обученные на огромных 
        объёмах текстовых данных. Они способны генерировать связный текст, отвечать


In [11]:
# Дополнительные примеры поиска
test_queries = [
    "Что такое машинное обучение?",
    "Как компьютеры понимают текст?",
    "Что такое глубокое обучение?",
    "Где применяется компьютерное зрение?"
]

print("\n" + "="*60)
print("ТЕСТИРОВАНИЕ ПОИСКА ПО РАЗНЫМ ЗАПРОСАМ")
print("="*60)

for query in test_queries:
    print(f"\n❓ Запрос: {query}")
    print("-" * 60)
    results = search_documents(query, top_k=2)
    for doc, score in results:
        print(f"  → {doc.metadata['topic']} (релевантность: {1 - score:.4f})")


ТЕСТИРОВАНИЕ ПОИСКА ПО РАЗНЫМ ЗАПРОСАМ

❓ Запрос: Что такое машинное обучение?
------------------------------------------------------------
  → машинное обучение (релевантность: 0.6509)
  → обучение с подкреплением (релевантность: 0.5242)

❓ Запрос: Как компьютеры понимают текст?
------------------------------------------------------------
  → нейронные сети (релевантность: 0.1301)
  → LLM (релевантность: 0.0634)

❓ Запрос: Что такое глубокое обучение?
------------------------------------------------------------
  → нейронные сети (релевантность: 0.3137)
  → машинное обучение (релевантность: -0.0858)

❓ Запрос: Где применяется компьютерное зрение?
------------------------------------------------------------
  → компьютерное зрение (релевантность: 0.3947)
  → компьютерное зрение (релевантность: -0.2822)


### 4.7. Генерация ответа с контекстом

Используем LLM для формирования ответа на основе найденного контекста.

In [12]:
!nvidia-smi

Wed Mar 18 15:39:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             28W /   70W |     621MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
!pip install bitsandbytes accelerate -q

# Загрузка модели с правильным CPU offload (ИСПРАВЛЕНО)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

llm_model = "Qwen/Qwen2.5-3B-Instruct"

print(f"Загрузка модели: {llm_model}")
print("Используется INT4 квантование + CPU offload...")

# Конфигурация INT4 квантования с CPU offload
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

# Загрузка токенизатора
summarizer_tokenizer = AutoTokenizer.from_pretrained(llm_model)

# Загрузка модели с квантованием и CPU offload
summarizer_model = AutoModelForCausalLM.from_pretrained(
    llm_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    offload_folder="offload",
    offload_state_dict=True,
)

print("✓ Модель загружена (INT4 + CPU offload)")
print(f"Устройство: {summarizer_model.device}")

Загрузка модели: Qwen/Qwen2.5-3B-Instruct
Используется INT4 квантование + CPU offload...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✓ Модель загружена (INT4 + CPU offload)
Устройство: cuda:0


In [14]:
# Функция RAG: поиск + генерация
def rag_query(question, top_k=2):
    """
    Полный RAG-пайплайн: поиск контекста и генерация ответа.
    
    Args:
        question: Вопрос пользователя
        top_k: Количество документов для поиска
    
    Returns:
        Сгенерированный ответ
    """
    # Поиск релевантных документов
    results = search_documents(question, top_k=top_k)
    
    # Объединение контекста
    context = " ".join([doc.page_content for doc, _ in results])
    
    # Формирование промпта
    prompt = f"""
    На основе следующего текста ответь на вопрос.
    
    КОНТЕКСТ:
    {context}
    
    ВОПРОС: {question}
    
    ОТВЕТ:
    """
    
    # Генерация ответа
    try:
        summary = generator(prompt, max_length=150, min_length=30, do_sample=False)
        return summary[0]['summary_text']
    except Exception as e:
        return f"Ошибка генерации: {e}. Контекст: {context[:200]}..."

print("RAG-функция готова")

RAG-функция готова


In [15]:
# Тестирование RAG
print("="*60)
print("ТЕСТИРОВАНИЕ RAG-СИСТЕМЫ")
print("="*60)

# Тестовые вопросы
questions = [
    "Что такое машинное обучение?",
    "Как работают нейронные сети?",
    "Что такое обработка естественного языка?"
]

for question in questions:
    print(f"\n❓ Вопрос: {question}")
    print("-"*60)
    
    # Поиск релевантных чанков
    docs = vectorstore.similarity_search(question, k=2)
    
    # Формирование контекста
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Формирование промпта
    prompt = f"""Контекст: {context}

Вопрос: {question}

Используй только информацию из контекста для ответа. Если ответа нет в контексте, скажи "Не могу найти ответ в предоставленной информации".

Ответ:"""
    
    # Генерация ответа с помощью T5
    try:
        # Токенизация
        inputs = summarizer_tokenizer(
            prompt,
            return_tensors="pt",
            max_length=512,
            truncation=True
        ).to(device)
        
        # Генерация
        outputs = summarizer_model.generate(
            inputs['input_ids'],
            max_length=150,
            min_length=30,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True,
            pad_token_id=summarizer_tokenizer.eos_token_id
        )
        
        # Декодирование
        answer = summarizer_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        print(f"💬 Ответ: {answer}")
        
    except Exception as e:
        print(f"💬 Ответ: Ошибка генерации: {e}. Контекст: {context[:200]}...")
    
    print("-"*60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


ТЕСТИРОВАНИЕ RAG-СИСТЕМЫ

❓ Вопрос: Что такое машинное обучение?
------------------------------------------------------------
💬 Ответ: Контекст: Машинное обучение — это подраздел искусственного интеллекта, 
        который изучает методы построения моделей, способных обучаться

Обучение с подкреплением — это тип машинного обучения, в котором агент 
        учится принимать решения, взаимодействуя со средой. Агент получает награды

Вопрос: Что такое машинное обучение?

Используй только информацию из контекста для ответа. Если ответа нет в контексте, скажи "Не могу найти ответ в предоставленной информации".

Ответ: Машинное обучение — это
------------------------------------------------------------

❓ Вопрос: Как работают нейронные сети?
------------------------------------------------------------
💬 Ответ: Ошибка генерации: Input length of input_ids is 165, but `max_length` is set to 150. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, se

### 4.8. Загрузка сохранённого индекса

Демонстрация загрузки ранее сохранённого индекса FAISS.

In [16]:
# Загрузка сохранённого индекса
print("Загрузка сохранённого индекса FAISS...")

loaded_vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Индекс загружен! Количество векторов: {loaded_vectorstore.index.ntotal}")

# Тестирование
test_query = "Что такое глубокое обучение?"
results = loaded_vectorstore.similarity_search_with_score(test_query, k=2)

print(f"\nТестовый запрос: '{test_query}'")
for doc, score in results:
    print(f"  → {doc.metadata['source']} (релевантность: {1 - score:.4f})")

Загрузка сохранённого индекса FAISS...
Индекс загружен! Количество векторов: 14

Тестовый запрос: 'Что такое глубокое обучение?'
  → neural_networks.txt (релевантность: 0.3137)
  → ml_intro.txt (релевантность: -0.0858)


### 4.9. Расширенный поиск с фильтрацией

Добавим фильтрацию по метаданным для более точного поиска.

In [17]:
# Поиск с фильтрацией по метаданным
def filtered_search(query, topic_filter=None, top_k=3):
    """
    Поиск с возможностью фильтрации по теме.
    
    Args:
        query: Поисковый запрос
        topic_filter: Фильтр по теме (или None)
        top_k: Количество результатов
    
    Returns:
        Список документов
    """
    if topic_filter:
        # Фильтрация через поиск по всем документам
        all_results = vectorstore.similarity_search_with_score(query, k=10)
        filtered = [(doc, score) for doc, score in all_results 
                    if doc.metadata.get('topic') == topic_filter]
        return filtered[:top_k]
    else:
        return vectorstore.similarity_search_with_score(query, k=top_k)

# Тестирование фильтрации
print("Поиск с фильтром по теме 'нейронные сети':")
results = filtered_search("как работает обучение", topic_filter="нейронные сети", top_k=2)

for doc, score in results:
    print(f"  → {doc.metadata['topic']}: {doc.page_content[:100]}...")

Поиск с фильтром по теме 'нейронные сети':
  → нейронные сети: и обработка естественного языка....
  → нейронные сети: скрытые слои и выходной слой. Глубокое обучение использует многоуровневые 
        нейронные сети дл...


---

## 5. Контрольные вопросы

**Ответы записывайте в эту ячейку (режим Markdown):**

1. **Какую проблему решает RAG и почему это важно?**
   > Ваш ответ...

2. **Что такое эмбеддинги и как они используются в поиске?**
   > Ваш ответ...

3. **Зачем нужно разбивать документы на чанки?**
   > Ваш ответ...

4. **Как работает векторный поиск в FAISS?**
   > Ваш ответ...

5. **Какие преимущества у RAG перед использованием LLM без контекста?**
   > Ваш ответ...

---

## 6. Полезные ресурсы

- 📚 [LangChain Documentation](https://python.langchain.com/docs/get_started/introduction) — официальная документация
- 📖 [FAISS Documentation](https://faiss.ai/) — библиотека векторного поиска
- 📖 [RAG Paper](https://arxiv.org/abs/2005.11401) — оригинальная статья RAG
- 🎥 [RAG Explained (YouTube)](https://www.youtube.com/watch?v=8rnXZQRQ7Fg) — визуальное объяснение
- 🔍 [Sentence Transformers](https://huggingface.co/sentence-transformers) — модели эмбеддингов

---

> **Примечание:** Все лабораторные работы выполняются в Google Colab с использованием бесплатных ресурсов. Сохраняйте копию ноутбука в своём Google Drive через `File → Save a copy in Drive`.